In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q25.pt
/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q50.pt
/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q75.pt
/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q10.pt


In [20]:
import os

# List all files in the attached dataset
dataset_path = '/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models'
print("Dataset contents:")
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        print(os.path.join(root, f))

Dataset contents:
/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q25.pt
/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q50.pt
/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q75.pt
/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models/dncnn_pq_q10.pt


In [21]:
!pip install -q fastapi uvicorn python-multipart pillow numpy opencv-python-headless scipy scikit-image torch torchvision

print("✅ All packages installed!")

✅ All packages installed!


In [27]:
import subprocess
import time
import os

# Kill any existing uvicorn processes
!pkill -f "uvicorn backend:app" 2>/dev/null

# Write backend to file first

backend_code = r'''import os
import io
import base64
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import cv2
from scipy.fftpack import dct, idct
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse

app = FastAPI()

# ─── CONFIG ───
MODEL_DIR = '/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ─── DnCNN MODEL ───
class DnCNN(nn.Module):
    def __init__(self, channels=1, num_layers=17):
        super(DnCNN, self).__init__()
        kernel_size = 3
        padding = 1
        features = 64
        layers = []
        layers.append(nn.Conv2d(channels, features, kernel_size, padding=padding, bias=True))
        layers.append(nn.ReLU(inplace=True))
        for _ in range(num_layers - 2):
            layers.append(nn.Conv2d(features, features, kernel_size, padding=padding, bias=False))
            layers.append(nn.BatchNorm2d(features))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Conv2d(features, channels, kernel_size, padding=padding, bias=False))
        self.dncnn = nn.Sequential(*layers)

    def forward(self, x):
        out = self.dncnn(x)
        return out

# ─── LOAD MODELS ───
models = {}
quality_map = {10: 'q10', 25: 'q25', 50: 'q50', 75: 'q75'}

def load_model(quality):
    q_key = quality_map.get(quality, 'q50')
    model_path = os.path.join(MODEL_DIR, f'dncnn_pq_{q_key}.pt')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")
    
    model = DnCNN(channels=1, num_layers=17).to(DEVICE)
    state_dict = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    return model

# ─── JPEG DCT COMPRESSION ───
def jpeg_compress(img_array, quality):
    """Apply JPEG-like DCT compression with quality factor."""
    img_float = img_array.astype(np.float32) - 128.0
    h, w = img_float.shape
    h_pad = (8 - h % 8) % 8
    w_pad = (8 - w % 8) % 8
    padded = np.pad(img_float, ((0, h_pad), (0, w_pad)), mode='edge')
    
    # DCT quantization matrix
    q_table = np.ones((8, 8)) * (100 - quality) / 50
    q_table[q_table < 1] = 1
    
    compressed = np.zeros_like(padded)
    for i in range(0, padded.shape[0], 8):
        for j in range(0, padded.shape[1], 8):
            block = padded[i:i+8, j:j+8]
            dct_block = dct(dct(block.T, norm='ortho').T, norm='ortho')
            quantized = np.round(dct_block / q_table) * q_table
            compressed[i:i+8, j:j+8] = idct(idct(quantized.T, norm='ortho').T, norm='ortho')
    
    result = compressed[:h, :w] + 128.0
    return np.clip(result, 0, 255).astype(np.uint8)

def apply_dncnn_denoising(img_array, quality):
    """Apply DnCNN denoising to compressed image."""
    model = load_model(quality)
    img_tensor = torch.from_numpy(img_array.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        denoised = model(img_tensor)
    
    denoised_np = denoised.squeeze().cpu().numpy() * 255.0
    return np.clip(denoised_np, 0, 255).astype(np.uint8)

# ─── METRICS ───
def compute_metrics(original, compressed):
    orig_float = original.astype(np.float64) / 255.0
    comp_float = compressed.astype(np.float64) / 255.0
    
    psnr_val = psnr(orig_float, comp_float, data_range=1.0)
    ssim_val = ssim(orig_float, comp_float, data_range=1.0)
    
    # Bits per pixel estimation
    bpp = 8 * (compressed.nbytes / original.size)
    file_size_kb = compressed.nbytes / 1024
    
    return {
        'psnr': float(psnr_val),
        'ssim': float(ssim_val),
        'bpp': float(bpp),
        'file_size_kb': float(file_size_kb)
    }

# ─── API ENDPOINTS ───
@app.get("/health")
def health():
    return {"status": "ok", "device": str(DEVICE), "models_available": list(quality_map.keys())}

@app.post("/compress")
async def compress(
    file: UploadFile = File(...),
    quality: int = Form(50),
    apply_dncnn: bool = Form(True),
    target_size: int = Form(512)
):
    try:
        # Read and preprocess image
        contents = await file.read()
        img = Image.open(io.BytesIO(contents)).convert('L')
        
        # Resize if needed
        img.thumbnail((target_size, target_size), Image.LANCZOS)
        img_array = np.array(img)
        
        # Compress
        compressed = jpeg_compress(img_array, quality)
        
        # Apply DnCNN if requested
        if apply_dncnn:
            compressed = apply_dncnn_denoising(compressed, quality)
        
        # Compute metrics
        metrics = compute_metrics(img_array, compressed)
        
        # Encode result
        result_img = Image.fromarray(compressed)
        buffer = io.BytesIO()
        result_img.save(buffer, format='PNG')
        img_b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
        
        return JSONResponse({
            "compressed_b64": img_b64,
            "metrics": metrics,
            "original_shape": img_array.shape,
            "quality": quality,
            "dncnn_applied": apply_dncnn
        })
        
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.get("/")
def root():
    return {"message": "DnCNN Nail Art Compression API", "endpoints": ["/health", "/compress"]}

# ─── MAIN ───
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('backend.py', 'w') as f:
    f.write(backend_code)

print("✅ backend.py written successfully!")
!wc -l backend.py




















✅ backend.py written successfully!
167 backend.py


In [30]:
import subprocess
import time
import urllib.request
import os
import signal

# Kill any existing uvicorn
for line in os.popen("ps aux | grep uvicorn | grep -v grep"):
    pid = int(line.split()[1])
    os.kill(pid, signal.SIGKILL)
    print(f"Killed old uvicorn PID {pid}")

time.sleep(1)

# Start uvicorn in background using Popen (no shell, no & needed)
proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "backend:app", "--host", "0.0.0.0", "--port", "8000", "--no-access-log"],
    stdout=open("/tmp/uvicorn.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print(f"🚀 Started uvicorn PID: {proc.pid}")
time.sleep(5)

# Show logs
print("\n--- Uvicorn Logs ---")
with open("/tmp/uvicorn.log", "r") as f:
    print(f.read())

# Health check
print("\n--- Health Check ---")
try:
    req = urllib.request.Request("http://localhost:8000/health", method="GET")
    resp = urllib.request.urlopen(req, timeout=10)
    data = resp.read().decode()
    print(f"✅ Backend is live!")
    print(f"🌐 http://localhost:8000")
    print(f"Response: {data}")
except Exception as e:
    print(f"❌ Backend not responding: {e}")
    print("\nFull logs:")
    with open("/tmp/uvicorn.log", "r") as f:
        print(f.read())

🚀 Started uvicorn PID: 204

--- Uvicorn Logs ---
INFO:     Started server process [204]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


--- Health Check ---
✅ Backend is live!
🌐 http://localhost:8000
Response: {"status":"ok","device":"cpu","models_available":[10,25,50,75]}


In [ ]:
import gradio as gr
import os
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from scipy.fftpack import dct, idct
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ─── CONFIG ───
MODEL_DIR = '/kaggle/input/datasets/mariyyyaaella/dncnn-nail-art-models'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ─── DnCNN MODEL (20 layers to match saved weights) ───
class DnCNN(nn.Module):
    def __init__(self, channels=1, num_layers=20):
        super(DnCNN, self).__init__()
        kernel_size = 3
        padding = 1
        features = 64
        layers = []
        layers.append(nn.Conv2d(channels, features, kernel_size, padding=padding, bias=True))
        layers.append(nn.ReLU(inplace=True))
        for _ in range(num_layers - 2):
            layers.append(nn.Conv2d(features, features, kernel_size, padding=padding, bias=False))
            layers.append(nn.BatchNorm2d(features))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Conv2d(features, channels, kernel_size, padding=padding, bias=False))
        self.dncnn = nn.Sequential(*layers)

    def forward(self, x):
        return self.dncnn(x)

# ─── LOAD MODELS ───
quality_map = {10: 'q10', 25: 'q25', 50: 'q50', 75: 'q75'}
models = {}

def load_model(quality):
    if quality in models:
        return models[quality]
    
    q_key = quality_map.get(quality, 'q50')
    model_path = os.path.join(MODEL_DIR, f'dncnn_pq_{q_key}.pt')
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")
    
    model = DnCNN(channels=1, num_layers=20).to(DEVICE)
    state_dict = torch.load(model_path, map_location=DEVICE)
    
    # Renommer net. → dncnn.
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key.replace('net.', 'dncnn.')
        new_state_dict[new_key] = value
    
    model.load_state_dict(new_state_dict, strict=False)
    model.eval()
    models[quality] = model
    print(f"✅ Loaded model Q{quality}")
    return model

# ─── JPEG DCT COMPRESSION (RGB) ───
def jpeg_compress_rgb(img_array, quality):
    """Compress each RGB channel separately."""
    result = np.zeros_like(img_array)
    for c in range(3):  # R, G, B
        channel = img_array[:, :, c].astype(np.float32) - 128.0
        h, w = channel.shape
        h_pad = (8 - h % 8) % 8
        w_pad = (8 - w % 8) % 8
        padded = np.pad(channel, ((0, h_pad), (0, w_pad)), mode='edge')
        
        q_table = np.ones((8, 8)) * (100 - quality) / 50
        q_table[q_table < 1] = 1
        
        compressed = np.zeros_like(padded)
        for i in range(0, padded.shape[0], 8):
            for j in range(0, padded.shape[1], 8):
                block = padded[i:i+8, j:j+8]
                dct_block = dct(dct(block.T, norm='ortho').T, norm='ortho')
                quantized = np.round(dct_block / q_table) * q_table
                compressed[i:i+8, j:j+8] = idct(idct(quantized.T, norm='ortho').T, norm='ortho')
        
        result[:, :, c] = np.clip(compressed[:h, :w] + 128.0, 0, 255)
    return result.astype(np.uint8)

def apply_dncnn_rgb(img_array, quality):
    """Apply DnCNN on each RGB channel separately."""
    model = load_model(quality)
    result = np.zeros_like(img_array)
    
    for c in range(3):
        ch = img_array[:, :, c].astype(np.float32) / 255.0
        tensor = torch.from_numpy(ch).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            denoised = model(tensor)
        result[:, :, c] = np.clip(denoised.squeeze().cpu().numpy() * 255.0, 0, 255)
    
    return result.astype(np.uint8)

# ─── METRICS (RGB) ───
def compute_metrics(original, compressed):
    orig_float = original.astype(np.float64) / 255.0
    comp_float = compressed.astype(np.float64) / 255.0
    
    # PSNR per channel then average
    psnr_vals = []
    ssim_vals = []
    for c in range(3):
        psnr_vals.append(psnr(orig_float[:, :, c], comp_float[:, :, c], data_range=1.0))
        ssim_vals.append(ssim(orig_float[:, :, c], comp_float[:, :, c], data_range=1.0))
    
    psnr_val = np.mean(psnr_vals)
    ssim_val = np.mean(ssim_vals)
    bpp = 8 * (compressed.nbytes / original.size)
    file_size_kb = compressed.nbytes / 1024
    
    return {
        'PSNR': f"{psnr_val:.2f} dB",
        'SSIM': f"{ssim_val:.4f}",
        'BPP': f"{bpp:.2f} bits/px",
        'Taille': f"{file_size_kb:.1f} KB"
    }

# ─── MAIN PROCESSING ───
def compress_image(input_image, quality_choice, use_dncnn):
    if input_image is None:
        return None, "Aucune image chargée", ""
    
    if isinstance(input_image, np.ndarray):
        img = Image.fromarray(input_image).convert('RGB')
    else:
        img = input_image.convert('RGB')
    
    img.thumbnail((512, 512), Image.LANCZOS)
    img_array = np.array(img)  # (H, W, 3) RGB
    
    quality_map_choice = {
        "Basique (Q25)": 25,
        "Économique (Q50)": 50,
        "Standard (Q75)": 75,
        "Premium (Q90)": 90
    }
    quality = quality_map_choice.get(quality_choice, 50)
    
    # Compress RGB
    compressed = jpeg_compress_rgb(img_array, quality)
    
    # Apply DnCNN per channel
    if use_dncnn:
        try:
            compressed = apply_dncnn_rgb(compressed, quality)
        except Exception as e:
            print(f"⚠️ DnCNN failed: {e}")
    
    metrics = compute_metrics(img_array, compressed)
    metrics_text = "\n".join([f"**{k}:** {v}" for k, v in metrics.items()])
    
    result_img = Image.fromarray(compressed)
    return result_img, metrics_text, f"Qualité: {quality_choice} | DnCNN: {'OUI' if use_dncnn else 'NON'}"

# ─── GRADIO INTERFACE ───
with gr.Blocks(title="💅 Nail Art Compression", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 💅 Nail Art Compression")
    gr.Markdown("Compressez vos images en **COULEUR** intelligemment avec DnCNN")
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(
                label="Image originale (RGB)",
                type="pil",
                image_mode="RGB",  # ← COULEUR
                height=400
            )
            
            quality_dropdown = gr.Dropdown(
                choices=["Basique (Q25)", "Économique (Q50)", "Standard (Q75)", "Premium (Q90)"],
                value="Économique (Q50)",
                label="Niveau de compression"
            )
            
            dncnn_checkbox = gr.Checkbox(
                label="Amélioration AI (DnCNN)",
                value=True,
                info="Réduction du bruit de compression par canal"
            )
            
            compress_btn = gr.Button("🚀 Compresser", variant="primary", size="lg")
        
        with gr.Column():
            output_img = gr.Image(
                label="Image compressée (RGB)",  # ← COULEUR
                height=400
            )
            
            status_text = gr.Textbox(
                label="Statut",
                interactive=False
            )
            
            metrics_box = gr.Markdown("### Métriques\n*En attente de compression...*")
    
    # Auto-compress
    input_img.change(
        fn=compress_image,
        inputs=[input_img, quality_dropdown, dncnn_checkbox],
        outputs=[output_img, metrics_box, status_text]
    )
    quality_dropdown.change(
        fn=compress_image,
        inputs=[input_img, quality_dropdown, dncnn_checkbox],
        outputs=[output_img, metrics_box, status_text]
    )
    dncnn_checkbox.change(
        fn=compress_image,
        inputs=[input_img, quality_dropdown, dncnn_checkbox],
        outputs=[output_img, metrics_box, status_text]
    )
    compress_btn.click(
        fn=compress_image,
        inputs=[input_img, quality_dropdown, dncnn_checkbox],
        outputs=[output_img, metrics_box, status_text]
    )

# Launch
demo.launch(share=True, debug=True)

Using device: cpu


/tmp/ipykernel_57/827661620.py:166: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="💅 Nail Art Compression", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://43bd7c5f4e599b2a21.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a par

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a param with shape torch.Size([1, 64, 3, 3]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).
⚠️ DnCNN failed: Error(s) in loading state_dict for DnCNN:
	size mismatch for dncnn.47.weight: copying a par